In [6]:
# Diagnóstico do computador onde este kernel Python está executando.
# Não instala dependências e não acessa credenciais.
import os
import platform
import shutil
import socket
import subprocess
from pathlib import Path

print("=== Ambiente do kernel ===")
print("Hostname:", socket.gethostname())
print("Sistema:", platform.platform())
print("Python:", platform.python_version())
print("Arquitetura:", platform.machine())
print("Diretório atual:", Path.cwd())
colab_runtime = any(os.environ.get(key) for key in
                    ("COLAB_RELEASE_TAG", "COLAB_BACKEND_VERSION", "COLAB_GPU"))
print("Indícios de runtime Colab:", "Sim" if colab_runtime else "Não detectados (não é conclusivo)")
print("Este diagnóstico descreve o kernel, mesmo quando a interface é o VS Code.")

print("\n=== CPU e RAM ===")
print("CPUs lógicas:", os.cpu_count())
cpu_info = Path("/proc/cpuinfo")
if cpu_info.exists():
    for line in cpu_info.read_text().splitlines():
        if line.startswith("model name"):
            print("CPU:", line.split(":", 1)[1].strip())
            break
mem_info = Path("/proc/meminfo")
if mem_info.exists():
    memory = {line.split(":", 1)[0]: int(line.split()[1])
              for line in mem_info.read_text().splitlines()
              if line.startswith(("MemTotal:", "MemAvailable:"))}
    for key, label in (("MemTotal", "RAM total"), ("MemAvailable", "RAM disponível")):
        if key in memory:
            print(f"{label}: {memory[key] / 1024**2:.2f} GiB")
else:
    print("RAM: diagnóstico disponível em runtimes Linux.")

print("\n=== GPU NVIDIA ===")
if shutil.which("nvidia-smi"):
    try:
        result = subprocess.run([
            "nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version",
            "--format=csv"], capture_output=True, text=True, timeout=15)
        if result.returncode == 0:
            print(result.stdout.strip())
            print("Memória em MiB; cada linha corresponde a uma GPU.")
        else:
            print("nvidia-smi encontrado, mas não conseguiu consultar a GPU.")
            print(result.stderr.strip())
    except (OSError, subprocess.TimeoutExpired) as exc:
        print("Falha ao consultar a GPU:", type(exc).__name__)
else:
    print("nvidia-smi não encontrado. GPU NVIDIA/CUDA não confirmada.")
    print("No Colab, selecione um ambiente com GPU antes de continuar.")


=== Ambiente do kernel ===
Hostname: eb4895c03e51
Sistema: Linux-6.6.122+-x86_64-with-glibc2.35
Python: 3.12.13
Arquitetura: x86_64
Diretório atual: /content
Indícios de runtime Colab: Sim
Este diagnóstico descreve o kernel, mesmo quando a interface é o VS Code.

=== CPU e RAM ===
CPUs lógicas: 12
CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
RAM total: 52.96 GiB
RAM disponível: 49.21 GiB

=== GPU NVIDIA ===
name, memory.total [MiB], memory.free [MiB], driver_version
NVIDIA L4, 23034 MiB, 22564 MiB, 580.82.07
Memória em MiB; cada linha corresponde a uma GPU.


# Qwen no Colab — API para o TCC2

Selecione **Ambiente de execução → Alterar tipo → GPU** e execute as células em ordem. Padrão: Qwen2.5-Coder-14B-Instruct, NF4; para testar a integração, selecione o 7B. Não troque o modelo automaticamente durante a coleta.

Por padrão, a configuração solicita os tokens por entrada oculta (`getpass`), compatível também com VS Code conectado à GPU do Colab. `COLAB_LLM_API_KEY` é um token que você escolhe (sua senha, não vazia), não uma chave OpenAI; `NGROK_AUTHTOKEN` vem da sua conta ngrok. Alternativamente, na interface web, defina `USE_COLAB_SECRETS = True` e crie os secrets `COLAB_LLM_API_KEY` (senha escolhida por você) e `NGROK_AUTHTOKEN` (conta ngrok) no painel de chaves do Colab e permita acesso a este notebook. Nunca salve os valores em células ou saídas. O túnel transporta prompts e respostas pelo ngrok.

Este notebook fornece `/v1/models` e `/v1/chat/completions`, sem streaming, com geração sequencial e JSON Schema via LM Format Enforcer. Não é uma implementação completa da API OpenAI. A sessão precisa permanecer ativa; não há mecanismos para evitar os limites do Colab.

**Estado da integração:** o cliente atual do repositório ainda precisa ser adaptado para URL/provedor, timeout e orçamento próprios. Apenas definir a URL no `.env` não altera o cliente existente. O exemplo ao final permite testar a API independentemente.


In [4]:
%pip install -q "transformers==4.51.3" "accelerate==1.6.0" "bitsandbytes==0.45.5" "lm-format-enforcer==0.10.11" "fastapi==0.115.12" "uvicorn==0.34.2" "pyngrok==7.2.5" "openai>=1.40,<2" "huggingface-hub>=0.30,<1"


Se o Colab solicitar reinicialização após instalar, reinicie a sessão e continue daqui. O PyTorch/CUDA é o fornecido pelo Colab; suas versões serão registradas no manifesto.

In [7]:
import os, json, time, secrets, threading, uuid, importlib.metadata
from pathlib import Path
import torch
from getpass import getpass
from huggingface_hub import model_info

MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct" # alternativa: Qwen/Qwen2.5-Coder-7B-Instruct
MODEL_REVISION = "main" # na coleta final, reutilize o SHA registrado no manifesto
SEED = 42
MAX_CONTEXT_TOKENS = 8192 # inclui entrada + saída; aumente apenas após medir VRAM
MAX_OUTPUT_TOKENS = 4096
PORT = 8000
# False funciona no navegador e no VS Code; entrada oculta, sem salvar tokens.
USE_COLAB_SECRETS = False # True apenas na interface web do Colab

def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if USE_COLAB_SECRETS:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except (userdata.TimeoutException, userdata.SecretNotFoundError, userdata.NotebookAccessError):
            print(f"Secret {name} indisponível; informe-o na entrada oculta abaixo.")
    return getpass(f"{name} (entrada oculta): ").strip()

API_KEY = read_secret("COLAB_LLM_API_KEY")
NGROK_TOKEN = read_secret("NGROK_AUTHTOKEN")
assert API_KEY, "Informe uma senha não vazia em COLAB_LLM_API_KEY."
assert NGROK_TOKEN, "Configure NGROK_AUTHTOKEN."
assert torch.cuda.is_available(), "Selecione um runtime com GPU."
REVISION = model_info(MODEL_ID, revision=MODEL_REVISION).sha
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print({"modelo": MODEL_ID, "revisao": REVISION, "gpu": torch.cuda.get_device_name(0),
       "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
       "dtype": str(DTYPE)})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


{'modelo': 'Qwen/Qwen2.5-Coder-14B-Instruct', 'revisao': 'aedcc2d42b622764e023cf882b6652e646b95671', 'gpu': 'NVIDIA L4', 'vram_gib': 22.03, 'dtype': 'torch.bfloat16'}


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION,
    quantization_config=quant, torch_dtype=DTYPE, device_map={"": 0}, attn_implementation="sdpa")
model.eval()
MANIFEST = {"model": MODEL_ID, "revision": REVISION, "quantization": "bitsandbytes-nf4-double",
    "dtype": str(DTYPE), "gpu": torch.cuda.get_device_name(0), "seed": SEED,
    "max_context_tokens": MAX_CONTEXT_TOKENS, "max_output_tokens": MAX_OUTPUT_TOKENS,
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "packages": {p: importlib.metadata.version(p) for p in
        ["transformers", "accelerate", "bitsandbytes", "lm-format-enforcer", "fastapi", "pyngrok"]}}
Path("manifesto_colab.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
print("Modelo carregado. Manifesto salvo sem credenciais.")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00001-of-00006.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Modelo carregado. Manifesto salvo sem credenciais.


A geração usa `do_sample=False`, equivalente à escolha gulosa; seed não garante reprodução bit a bit entre GPUs. JSON Schema restringe a sintaxe, mas a validade da correção continua sendo avaliada pelo pipeline. O limite de contexto causa erro explícito, sem truncar prompts. Falta de VRAM retorna erro; reduza o contexto ou selecione explicitamente o 7B em uma nova execução.

In [9]:
from typing import Literal
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from pydantic import BaseModel, ConfigDict, Field

app = FastAPI(docs_url=None, redoc_url=None, openapi_url=None)
auth = HTTPBearer()
generation_lock = threading.Lock()

def authorize(credentials: HTTPAuthorizationCredentials = Depends(auth)):
    if not secrets.compare_digest(credentials.credentials, API_KEY):
        raise HTTPException(401, "Token inválido")

class Message(BaseModel):
    model_config = ConfigDict(extra="forbid")
    role: Literal["system", "user", "assistant"]
    content: str

class Completion(BaseModel):
    model_config = ConfigDict(extra="forbid")
    model: str
    messages: list[Message] = Field(min_length=1, max_length=100)
    temperature: float = 0
    seed: int = SEED
    max_completion_tokens: int = Field(default=MAX_OUTPUT_TOKENS, ge=1, le=MAX_OUTPUT_TOKENS)
    response_format: dict | None = None
    stream: bool = False

@app.get("/v1/models", dependencies=[Depends(authorize)])
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "Qwen"}]}

@app.get("/health", dependencies=[Depends(authorize)])
def health():
    return {"status": "ok", "manifest": MANIFEST}

@app.post("/v1/chat/completions", dependencies=[Depends(authorize)])
def complete(req: Completion):
    if req.model != MODEL_ID:
        raise HTTPException(400, "Modelo diferente do carregado")
    if req.stream or req.temperature != 0:
        raise HTTPException(400, "Use stream=false e temperature=0")
    schema = None
    if req.response_format is not None:
        fmt = req.response_format
        if fmt.get("type") != "json_schema":
            raise HTTPException(400, "response_format suporta somente json_schema")
        schema = fmt.get("json_schema", {}).get("schema")
        if not isinstance(schema, dict):
            raise HTTPException(400, "JSON Schema ausente")
    if not generation_lock.acquire(blocking=False):
        raise HTTPException(429, "GPU ocupada; configure K=1", headers={"Retry-After": "5"})
    try:
        inputs = tokenizer.apply_chat_template([m.model_dump() for m in req.messages],
            add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
        n_input = inputs["input_ids"].shape[-1]
        if n_input + req.max_completion_tokens > MAX_CONTEXT_TOKENS:
            raise HTTPException(400, "Entrada + saída excedem MAX_CONTEXT_TOKENS; prompt não foi truncado")
        options = {}
        if schema is not None:
            try:
                options["prefix_allowed_tokens_fn"] = build_transformers_prefix_allowed_tokens_fn(
                    tokenizer, JsonSchemaParser(schema))
            except Exception as exc:
                raise HTTPException(400, "Schema incompatível com o decodificador") from exc
        set_seed(req.seed)
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=req.max_completion_tokens,
                do_sample=False, pad_token_id=tokenizer.eos_token_id, **options)
        generated = output[0, n_input:]
        content = tokenizer.decode(generated, skip_special_tokens=True)
        eos_ids = model.generation_config.eos_token_id
        eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]
        reason = "stop" if generated[-1].item() in eos_ids else "length"
        n_output = len(generated)
        return {"id": "chatcmpl-" + uuid.uuid4().hex, "object": "chat.completion",
            "created": int(time.time()), "model": MODEL_ID,
            "system_fingerprint": "qwen-" + REVISION[:12] + "-nf4",
            "choices": [{"index": 0, "message": {"role": "assistant", "content": content,
                "refusal": None}, "finish_reason": reason}],
            "usage": {"prompt_tokens": n_input, "completion_tokens": n_output,
                "total_tokens": n_input + n_output}}
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise HTTPException(503, "VRAM insuficiente para esta configuração") from exc
    finally:
        generation_lock.release()


In [10]:
import uvicorn, requests
from pyngrok import ngrok

if "server" in globals() and server.started:
    raise RuntimeError("Servidor já ativo. Execute a célula de encerramento antes de reiniciar.")
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, access_log=False))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
for _ in range(100):
    if server.started:
        break
    if not server_thread.is_alive():
        raise RuntimeError("Falha ao iniciar servidor")
    time.sleep(0.1)
assert server.started, "Servidor não iniciou dentro do prazo"
assert requests.get(f"http://127.0.0.1:{PORT}/health",
    headers={"Authorization": f"Bearer {API_KEY}"}, timeout=10).status_code == 200
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT, "http", bind_tls=True)
BASE_URL = tunnel.public_url + "/v1"
print("URL da API:", BASE_URL)
print("Use o token do secret COLAB_LLM_API_KEY no cliente; ele não será exibido.")


INFO:     Started server process [470]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


URL da API: https://uphold-suitor-scored.ngrok-free.dev/v1                                          
Use o token do secret COLAB_LLM_API_KEY no cliente; ele não será exibido.


Teste curto pelo túnel. Não é uma tentativa experimental. A primeira geração pode demorar. Só inicie a coleta após conferir também uma resposta com o schema real do projeto.

In [11]:
from openai import OpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=600, max_retries=0)
assert client.models.list().data[0].id == MODEL_ID
smoke_schema = {"type": "object", "properties": {"ok": {"type": "boolean"}},
                "required": ["ok"], "additionalProperties": False}
result = client.chat.completions.create(model=MODEL_ID, temperature=0, seed=SEED,
    max_completion_tokens=32, messages=[{"role": "user", "content": 'Responda com um objeto JSON com ok igual a true.'}],
    response_format={"type": "json_schema", "json_schema": {
        "name": "smoke", "strict": True, "schema": smoke_schema}})
assert result.choices[0].finish_reason == "stop", "Resposta truncada"
assert json.loads(result.choices[0].message.content) == {"ok": True}
print("Teste de autenticação, túnel e JSON concluído.")


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:653: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


Teste de autenticação, túnel e JSON concluído.


## Testar a partir do computador

Use a URL exibida e o mesmo token, fornecidos por variáveis de ambiente, sem gravar credenciais no código:

```python
import os
from openai import OpenAI
client = OpenAI(base_url=os.environ["COLAB_LLM_BASE_URL"],
                api_key=os.environ["COLAB_LLM_API_KEY"], timeout=600)
print(client.models.list())
```

A adaptação do pipeline deverá enviar `model`, `temperature=0`, `seed`, `max_completion_tokens` e o `response_format` de `src/pipeline/llm/schema.py`. Configure K=1 e timeout compatível com gerações longas. Este servidor não informa preço de API; contabilize o custo do Colab separadamente e não adicione esse modelo à tabela de preços da OpenAI.

Baixe o manifesto e preserve-o com os resultados. Ele não contém tokens. Não salve saídas com credenciais no Git. Para a coleta final, fixe `MODEL_REVISION` no SHA do piloto; versões de bibliotecas e GPU também influenciam os resultados.


In [ ]:
from google.colab import files
files.download("manifesto_colab.json")


## Encerrar
Execute apenas após concluir o experimento. A URL muda quando um novo túnel é criado.

In [ ]:
if "tunnel" in globals():
    ngrok.disconnect(tunnel.public_url)
if "server" in globals():
    server.should_exit = True
if "server_thread" in globals():
    server_thread.join(timeout=10)
print("Encerramento solicitado. Desconecte o runtime para liberar a GPU.")


## Referências
- [Qwen2.5-Coder-14B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-14B-Instruct)
- [Quantização bitsandbytes](https://huggingface.co/docs/transformers/v4.46.3/en/quantization/bitsandbytes)
- [LM Format Enforcer](https://github.com/noamgat/lm-format-enforcer)
- [pyngrok no Colab](https://pyngrok.readthedocs.io/en/stable/integrations.html)
- [Limites do Colab](https://research.google.com/colaboratory/faq.html)
